# RIM Weighting (v 1.0)

Este script usa la libreria Weightipy para ejecutar ponderaciones por el algoritmo RIM.

~~~
py -m pip install weightipy
~~~

Documentación oficial de Weightypy: https://pypi.org/project/weightipy/

Autor: Fernando Monrroy

In [1]:
# Inicialización 
import weightipy as wp
import pandas as pd
from pyreadstat import write_sav

In [2]:
# Importar dataset 
df = pd.read_spss('./Data/ZAD3SVX_Stacked_FINAL.sav')

# Filtro correspondiente
df = df[(df['Pais']=='México') & (df['Variable_Type']=='Marca')]

### Exploración de las variables y demográficos 

Explorar las variables demograficas a ponderar, entender como estan agrupados los cortes y generar agrupaciones que correspondan con nuestros targets de ponderación como sea necesario. 

El engine de ponderación requiere uqe las variables que utilicemos para ponderar sean categoricas o numericas, los objetos que se crean desde Pandas son por defecto dtype: Object, es necesario transformarlos a categorical al crearlos con la función 
~~~
pd.Categorical("La serie va aquí")
~~~

In [3]:
#Exploración de distribucion sin ponderar 
print(df['SEXQUOTA'].value_counts(normalize=True))

print(df['AGEQUOTA'].value_counts(normalize=True))

print(df['NSE_MEXICO'].value_counts(normalize=True))

SEXQUOTA
Mujer     0.585518
Hombre    0.414482
Name: proportion, dtype: float64
AGEQUOTA
28-42    0.451935
43-55    0.259675
18-27    0.235955
16-17    0.052434
Name: proportion, dtype: float64
NSE_MEXICO
D+     0.232210
C      0.209738
C+     0.192260
C-     0.132335
A/B    0.123596
D      0.109863
Name: proportion, dtype: float64


In [4]:
# preparación de netos NSE 
nse_dict = {
    'A/B':'ABC+',
    'C+': 'ABC+',
    'C': 'CC-',
    'C-': 'CC-',
    'D+': 'D+D',
    'D': 'D+D',
    'E': pd.NA,
}

df['NSE_MEXICO_NET'] = pd.Categorical(df['NSE_MEXICO'].apply(lambda x: nse_dict[x]))

### Ponderación - Definición del schema

**Paso 1**: Definir el target. Se requiere crear una lista que contiene un diccionario por variable a controlar. 
Dentro de cada diccionario por variable un subdiccionario con las categorias y los pesos. 

***Nota**: Los pesos deben sumar exactamente 100*

**Paso 2**: Si necesitamos ponderar por subgrupos, ejemplo, por olas habra que crear un grupo y aplicar definir en el esquema el target para cada uno. 

Aplicabilidad de subgrupos:
- Podemos usar subrgupos pasa aplicar el ponderador por cada ola del estudio, e incluso asegurarnos que las olas tengan el mismo peso a lo largo del tiempo. 
- Otro uso alternativo de subgrupos es para ponderaciones cruzadas, donde las distribución de ciertos demograficos cambie por regiones. 


In [5]:
# Inicializar el objeto del esquema de ponderación 
scheme = wp.Rim(name='RIM')

# Definir los targets de ponderación 
target = [
    {
        'AGEQUOTA':{
            '16-17':5.3,
            '18-27':26.7,
            '28-42':36.9,
            '43-55':31.1
        }
    },
    {
        'SEXQUOTA':{
            'Hombre':43,
            'Mujer':57
        }
    }, 
    {
        'NSE_MEXICO_NET':{
            'ABC+':12.1,
            'CC-':42.4,
            'D+D':45.5
        }
    }
]

# Definir los grupos, podemos definir un solo target para todos los grupos o diferentes targets para cada grupo. 
# La definición del filtro es un string que se pasa a la función pd.Query(), ver documentación de Pandas en caso de dudas. 
scheme.add_group(name='W1', filter_def='WAVE == \"Ola 1\"', targets=target)
#scheme.add_group(name='W2', filter_def='WAVE == \"Ola 2\"', targets=target)

# Peso de cada grupo, si se define controla el peso total de cada grupo, si no se deja a la distribución natural 
group_targets = {
    'W1': 100
}

# Aplicar los pesos de cada grupo al schema 
scheme.group_targets(group_targets)

### Ponderación - Generar el ponderador 

Pray and run :D 

In [6]:
# Inicialización del engine 
engine = wp.WeightEngine(data=df)

# Adición del eschema al engine, key=ID Variable 
engine.add_scheme(scheme=scheme, key='Respondent_Serial', verbose=True)

# Corremos la ponderación y guardamos el resultado en otro DF 
engine.run()
df_weighted = engine.dataframe()

In [7]:
# Reporte de ponderación, verificar eficiencia, factor promedio y el minimo y maximo. Rangos ideales kantar mean weight factor>0.6 y  mean weight factor<1.4 
engine.get_report()

Weight variable,weights_RIM
Weight group,W1
Weight filter,"WAVE == ""Ola 1"""
Total: unweighted,34443.000000
Total: weighted,34443.000000
Weighting efficiency,82.548273
Iterations required,18.000000
Mean weight factor,1.000000
Minimum weight factor,0.318367
Maximum weight factor,1.732232
Weight factor ratio,5.440986


### Verificación de resultados 

In [8]:
print(df_weighted.groupby('SEXQUOTA', observed=True)['weights_RIM'].sum() / df_weighted.shape[0])

print(df_weighted.groupby('AGEQUOTA', observed=True)['weights_RIM'].sum() / df_weighted.shape[0])

print(df_weighted.groupby('NSE_MEXICO_NET', observed=True)['weights_RIM'].sum() / df_weighted.shape[0])

SEXQUOTA
Hombre    0.43
Mujer     0.57
Name: weights_RIM, dtype: float64
AGEQUOTA
16-17    0.053
18-27    0.267
28-42    0.369
43-55    0.311
Name: weights_RIM, dtype: float64
NSE_MEXICO_NET
ABC+    0.121
CC-     0.424
D+D     0.455
Name: weights_RIM, dtype: float64


### Output

In [9]:
write_sav(df_weighted, './Data/ZAD3SVX_Stacked_FINAL Weighted.sav')

In [14]:
df_weighted[['Respondent_Serial','weights_RIM']].to_csv('./Data/Weight.csv', index=False)